In [22]:
# 1. Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Input

In [ ]:
# 2. Load Train and Test Datasets
train_df = pd.read_csv("trainset.csv")
test_df = pd.read_csv("testset.csv")

print("Training Data:")
print(train_df.head())
print(f"\nTraining data shape: {train_df.shape}")

print("\n\nTest Data:")
print(test_df.head())
print(f"Test data shape: {test_df.shape}")

In [ ]:
# 3. Prepare and Scale Data
# Extract Open prices from both datasets
train_data = train_df['Open'].values.reshape(-1, 1)
test_data = test_df['Open'].values.reshape(-1, 1)

# Scale the data
scaler = MinMaxScaler(feature_range=(0, 1))
train_data_scaled = scaler.fit_transform(train_data)
test_data_scaled = scaler.transform(test_data)

print(f"Train data scaled shape: {train_data_scaled.shape}")
print(f"Test data scaled shape: {test_data_scaled.shape}")

In [ ]:
# 4. Create Time Series Dataset (use past 60 days to predict next day)
def create_dataset(dataset):
    X, y = [], []
    for i in range(60, len(dataset)):
        X.append(dataset[i-60:i, 0])
        y.append(dataset[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_dataset(train_data_scaled)
X_test, y_test = create_dataset(test_data_scaled)

# Reshape for RNN: [samples, timesteps, features]
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 5. Build RNN Model
model = Sequential()
model.add(Input(shape=(60, 1)))
model.add(SimpleRNN(50, return_sequences=True))
model.add(SimpleRNN(50))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

In [ ]:
# 6. Train RNN Model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

In [ ]:
# 7. Make Predictions
predicted = model.predict(X_test)

# Convert back to original scale
predicted = scaler.inverse_transform(predicted)
real = scaler.inverse_transform(y_test.reshape(-1, 1))

print(f"Predicted shape: {predicted.shape}")
print(f"Real shape: {real.shape}")

In [ ]:
# 8. Calculate Performance Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math

mse = mean_squared_error(real, predicted)
rmse = math.sqrt(mse)
mae = mean_absolute_error(real, predicted)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")

In [ ]:
# 9. Plot Results
plt.figure(figsize=(14, 5))

# Plot 1: Predictions vs Real
plt.subplot(1, 2, 1)
plt.plot(real, color='red', label='Real Stock Price', linewidth=2)
plt.plot(predicted, color='blue', label='Predicted Stock Price', linewidth=2)
plt.title("Google Stock Price Prediction using RNN")
plt.xlabel("Time")
plt.ylabel("Stock Price")
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Training History
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title("Model Loss During Training")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()